# GameTheory-04j — Rationalisabilité : ce que la seule rationalité peut délier

Le carnet [GameTheory-04-NashEquilibrium-Python.ipynb](GameTheory-04-NashEquilibrium-Python.ipynb) calcule des équilibres de Nash en stratégies mixtes. L'équilibre exige une **cohérence mutuelle** : la stratégie de chaque joueur est une meilleure réponse à celle de l'autre, et réciproquement. Que reste-t-il si on abandonne cette exigence et qu'on ne garde que la **rationalité** — chaque joueur joue une meilleure réponse à *une* croyance sur son adversaire — itérée par connaissance commune ? La réponse est l'ensemble des **stratégies rationalisables** : ce carnet le construit, le calcule, et mesure l'écart exact qui le sépare des équilibres du palier 04.

Trois idées structurent le parcours :

1. une stratégie peut être **condamnée sans être dominée par une stratégie pure** — la Table 5.8 de Bonanno livre un exemple qu'on vérifie au calcul ;
2. le **théorème de Pearce (1984)** — jamais-meilleure-réponse *si et seulement si* dominée par une stratégie mixte — est vérifié numériquement **dans les deux sens**, par programmation linéaire ;
3. l'**IDSDS cardinal** (itération de la suppression) définit les stratégies rationalisables ; sur les jeux mêmes du palier 04 (Dilemme du Prisonnier, Battle of the Sexes, chasse au cerf, Matching Pennies), on mesure que l'inclusion dans les équilibres est **stricte dès qu'aucune dominance n'amorce la purge**.

**Position dans la série** : accrétion du palier 04 — la rationalisabilité est la question « que délie la seule rationalité, sans équilibre ? » que le calcul des mixtes suppose tranchée. Le futur carnet 04i (Utilité Espérée et Risque) construira les utilités cardinales que la dominance par stratégies mixtes requiert ; [GameTheory-04f-Theories-Decision-Predicteur-Python.ipynb](GameTheory-04f-Theories-Decision-Predicteur-Python.ipynb) lit les mixtes comme des croyances ; [GameTheory-02d-Epistemique-Connaissance-Commune-Python.ipynb](GameTheory-02d-Epistemique-Connaissance-Commune-Python.ipynb) construit les opérateurs de connaissance que la lecture épistémique (section 5) consomme. Source principale : Bonanno (2015), ch. 5.4 « Strict dominance and rationalizability » (p. 201-204), lu intégralement sur la copie archivée du gisement (`G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\`).

## Références

**Bonanno (2015)**, *Game Theory*, ch. 5.4 « Strict dominance and rationalizability » : la colonne vertébrale de ce carnet — Table 5.8 (dominance par une stratégie mixte), théorème 5.3 [Pearce, 1984], définition 5.6 (IDSDS cardinal), Figure 5.9 (l'exemple en quatre suppressions, rejoué cellule par cellule en section 3), remarques 5.5-5.6, exercices 5.12-5.14 (repris en section 7). Les jeux de la section 4 sont ceux du palier 04. Gisement cluster : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\`.

In [1]:
import numpy as np
import itertools
import nashpy as nash
from scipy.optimize import linprog
from fractions import Fraction

print("numpy", np.__version__, "| nashpy", nash.__version__)

numpy 2.2.6 | nashpy 0.0.43


## 1. Dominée par une stratégie mixte : ce que l'IDSDS ordinal ne voit pas

Le chapitre 1 du manuel définit l'**IDSDS ordinal** : supprimer itérativement les stratégies strictement dominées **par une autre stratégie pure**. Avec des paiements cardinaux (construits au carnet 04i), une stratégie peut être condamnée — jamais jouée par un joueur rationnel — sans être dominée par aucune stratégie pure. La Table 5.8 de Bonanno en est le témoin, avec les paiements du Joueur 1 en premier :

| | D | E |
|---|---|---|
| **A** | 0 , 1 | 4 , 0 |
| **B** | 1 , 2 | 1 , 4 |
| **C** | 2 , 0 | 0 , 1 |

In [2]:
# Table 5.8 de Bonanno (p. 201) : lignes A, B, C (Joueur 1) ; colonnes D, E (Joueur 2).
# Paiements du Joueur 1 :
T58_P1 = np.array([[0.0, 4.0],
                   [1.0, 1.0],
                   [2.0, 0.0]])
labels_P1 = ["A", "B", "C"]
labels_P2 = ["D", "E"]

# (a) B est-il domine par une strategie PURE ? A domine B ? C domine B ?
for dom_pur in [0, 2]:
    ecart = T58_P1[dom_pur] - T58_P1[1]
    print(f"{labels_P1[dom_pur]} - B (ecart par colonne) : {ecart}  ->  domine strictement : {bool(np.all(ecart > 0))}")

# (b) Le gain de B contre une croyance quelconque q sur (D, E) :
q = Fraction(1, 3)   # croyance arbitraire, le resultat n'en dependra pas
gain_B = 1 * q + 1 * (1 - q)
print(f"gain de B contre q = {q} sur (D,E) : {gain_B} (constant : B rapporte 1 dans les deux colonnes)")

# (c) La strategie mixte (1/3 A, 0 B, 2/3 C) du manuel :
mixte = np.array([1/3, 0.0, 2/3])
gain_mixte = mixte @ T58_P1
print(f"gain de la mixte (1/3 A + 2/3 C) contre D : {gain_mixte[0]:.6f} (= 4/3)")
print(f"gain de la mixte (1/3 A + 2/3 C) contre E : {gain_mixte[1]:.6f} (= 4/3)")
print(f"marge minimale sur B : {np.min(gain_mixte - T58_P1[1]):.6f} (> 0 : B est strictement domine par cette mixte)")

A - B (ecart par colonne) : [-1.  3.]  ->  domine strictement : False
C - B (ecart par colonne) : [ 1. -1.]  ->  domine strictement : False
gain de B contre q = 1/3 sur (D,E) : 1 (constant : B rapporte 1 dans les deux colonnes)
gain de la mixte (1/3 A + 2/3 C) contre D : 1.333333 (= 4/3)
gain de la mixte (1/3 A + 2/3 C) contre E : 1.333333 (= 4/3)
marge minimale sur B : 0.333333 (> 0 : B est strictement domine par cette mixte)


### Interprétation : condamnée sans être dominée à l'œil nu

La lecture des sorties dit exactement le phénomène : ni A ni C ne domine B (le premier bloc rend un `False` sur les deux colonnes), le gain de B est **constant égal à 1** quelle que soit la croyance de l'adversaire, et la mixte $
rac{1}{3}A + 
rac{2}{3}C$ rapporte $
rac{4}{3}$ contre **les deux** colonnes — une marge de $
rac{1}{3}$ sur B, partout. Autrement dit : quel que soit ce que le Joueur 1 croit que le Joueur 2 fera, jouer B est strictement irrationnel, alors qu'aucune stratégie pure ne le dominatione. C'est l'information **cardinale** (les valeurs 0, 1, 2, 4, pas seulement leur ordre) qui rend la condamnation visible : c'est le pont direct vers le futur carnet 04i sur les utilités espérées.

## 2. Le théorème de Pearce (1984), vérifié dans les deux sens

**Théorème 5.3 [Pearce, 1984]** (énoncé dans Bonanno, p. 201) : dans un jeu à deux joueurs en forme stratégique avec paiements cardinaux, une stratégie pure $s_i$ du Joueur $i$ n'est la meilleure réponse à **aucune** stratégie mixte de l'adversaire **si et seulement si** $s_i$ est strictement dominée par une stratégie mixte du Joueur $i$.

Les deux membres sont des problèmes de programmation linéaire, qu'on implémente indépendamment :

- **jamais-meilleure-réponse** : existe-t-il une croyance $q$ (distribution sur les colonnes) telle que $u(s, q) \geq u(s', q)$ pour toute autre ligne $s'$ ? On maximise la marge $t$ du système $\sum_j q_j\,(u(s,j) - u(s',j)) \geq t$ ; si le maximum est $< 0$, aucune croyance ne fait de $s$ une meilleure réponse.
- **dominée par une mixte** : existe-t-il une mixte $p$ sur les lignes telle que $\sum_i p_i\,u(i,j) \geq u(s,j) + t$ pour toute colonne $j$, avec $t > 0$ ?

In [3]:
def marge_meilleure_reponse(M, s):
    """Joueur 1 (M = gains des lignes) : max t tel que u(s,q) - u(s',q) >= t pour tout s', q une distribution sur les colonnes.
    Retourne (t_opt, q). t_opt >= 0  <=>  s est meilleure reponse a la croyance q (egalites permises)."""
    M = np.asarray(M, dtype=float)
    n_rows, n_cols = M.shape
    autres = [i for i in range(n_rows) if i != s]
    # variables : q (n_cols), t (1) ; maximiser t = minimiser -t
    c = np.zeros(n_cols + 1); c[-1] = -1.0
    A_ub, b_ub = [], []
    for i in autres:
        ligne = np.concatenate([M[i] - M[s], [1.0]])
        A_ub.append(ligne); b_ub.append(0.0)      # (u(i)-u(s)) . q + t <= 0  <=>  u(s,q) - u(i,q) >= t
    A_eq = [np.concatenate([np.ones(n_cols), [0.0]])]; b_eq = [1.0]
    bornes = [(0, None)] * n_cols + [(None, None)]
    res = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                  A_eq=np.array(A_eq), b_eq=np.array(b_eq), bounds=bornes, method="highs")
    return res.x[-1], res.x[:n_cols]

def marge_dominee_par_mixte(M, s):
    """Joueur 1 : max t tel que (p . M)(j) >= M[s,j] + t pour toute colonne j, p une distribution sur les lignes.
    Retourne (t_opt, p). t_opt > 0  <=>  s est strictement dominee par la mixte p."""
    M = np.asarray(M, dtype=float)
    n_rows, n_cols = M.shape
    c = np.zeros(n_rows + 1); c[-1] = -1.0
    A_ub, b_ub = [], []
    for j in range(n_cols):
        ligne = np.concatenate([-M[:, j], [1.0]])   # -p.M(:,j) + t <= -M[s,j]
        A_ub.append(ligne); b_ub.append(-M[s, j])
    A_eq = [np.concatenate([np.ones(n_rows), [0.0]])]; b_eq = [1.0]
    bornes = [(0, None)] * n_rows + [(None, None)]
    res = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                  A_eq=np.array(A_eq), b_eq=np.array(b_eq), bounds=bornes, method="highs")
    return res.x[-1], res.x[:n_rows]

print("Table 5.8, verification des deux sens de Pearce pour chaque ligne :")
print(f"{'ligne':>5} | {'marge meilleure reponse':>24} | {'jamais MR':>9} | {'marge dominee mixte':>20} | {'dominee':>8} | accord")
for s in range(3):
    t1, _ = marge_meilleure_reponse(T58_P1, s)
    t2, p = marge_dominee_par_mixte(T58_P1, s)
    jamais_mr = t1 < -1e-9
    dominee = t2 > 1e-9
    print(f"{labels_P1[s]:>5} | {t1:>24.6f} | {str(jamais_mr):>9} | {t2:>20.6f} | {str(dominee):>8} | {jamais_mr == dominee}")
print()
print("Mixte dominatrice de B retrouvee par LP :", np.round(marge_dominee_par_mixte(T58_P1, 1)[1], 6), "(attendu : 1/3, 0, 2/3)")

Table 5.8, verification des deux sens de Pearce pour chaque ligne :
ligne |  marge meilleure reponse | jamais MR |  marge dominee mixte |  dominee | accord
    A |                 3.000000 |     False |            -0.000000 |    False | True
    B |                -0.333333 |      True |             0.333333 |     True | True
    C |                 1.000000 |     False |            -0.000000 |    False | True

Mixte dominatrice de B retrouvee par LP : [0.333333 0.       0.666667] (attendu : 1/3, 0, 2/3)


### Interprétation : deux questions différentes, la même réponse

Les trois lignes d'accord `True` disent le théorème : pour **chaque** ligne de la Table 5.8, « aucune croyance n'en fait une meilleure réponse » (marge optimale négative) coïncide exactement avec « une stratégie mixte la domine strictement » (marge optimale positive) — et le solveur **retrouve de lui-même** la mixte $(
rac{1}{3}, 0, 
rac{2}{3})$ du manuel. La coïncidence n'est pas un hasard de cet exemple : c'est l'équivalence de Pearce. Le lien conceptuel : chez le Joueur 1, une *croyance* sur l'adversaire et une *stratégie mixte* de l'adversaire sont le même objet mathématique (une distribution) — c'est spécifique aux deux joueurs, et l'exercice 3 (du manuel 5.14) montrera qu'à trois joueurs une croyance peut être **corrélée**, ce qu'un profil mixte indépendant ne peut pas être.

## 3. IDSDS cardinal et stratégies rationalisables

**Définition 5.6 (Cardinal IDSDS)** : supprimer du jeu, pour chaque joueur, les stratégies strictement dominées **par une stratégie mixte** de ce joueur ; réitérer sur le jeu réduit ; le survivant $G^\infty$ porte les **stratégies rationalisables**. La Figure 5.9 du manuel (p. 203) déroule la procédure sur un jeu $3 \times 3$ — on la rejoue avec le solveur LP, suppression par suppression :

| | D | E | F |
|---|---|---|---|
| **A** | 3 , 4 | 2 , 1 | 1 , 2 |
| **B** | 0 , 0 | 1 , 3 | 4 , 1 |
| **C** | 1 , 4 | 1 , 4 | 2 , 6 |

In [4]:
# Figure 5.9 (i) de Bonanno : le jeu G0.
G0_P1 = np.array([[3.0, 2.0, 1.0],
                  [0.0, 1.0, 4.0],
                  [1.0, 1.0, 2.0]])
G0_P2 = np.array([[4.0, 1.0, 2.0],
                  [0.0, 3.0, 1.0],
                  [4.0, 4.0, 6.0]])

def idsds_cardinal_trace(P1, P2, lignes, colonnes):
    """Itere la suppression des strategies strictement dominees PAR UNE MIXTE, chaque joueur a son tour.
    Retourne la trace detaillee et les index survivants (lignes, colonnes)."""
    P1, P2 = P1.copy(), P2.copy()
    lignes, colonnes = list(lignes), list(colonnes)
    trace = []
    etape = 0
    while True:
        supprime = False
        for (M, vivants, noms, joueur) in [
            (P1[np.ix_(lignes, colonnes)], lignes, noms_lignes_global, "Joueur 1"),
            (P2[np.ix_(lignes, colonnes)].T, colonnes, noms_colonnes_global, "Joueur 2"),
        ]:
            # Pour le Joueur 2 on transpose : lignes de M = strategies de P2, colonnes = strategies de P1
            for k in range(M.shape[0]):
                t, p = marge_dominee_par_mixte(M, k)
                if t > 1e-9:
                    mixte_txt = {noms[v]: round(float(x), 4) for v, x in zip(vivants, p)}
                    trace.append(f"etape {etape} : {joueur} supprime {noms[vivants[k]]}"
                                 f" (dominee par la mixte {mixte_txt}, marge {t:.4f})")
                    vivants.pop(k)
                    supprime = True
                    break
            if supprime:
                break
        if not supprime:
            break
        etape += 1
    return trace, lignes, colonnes

noms_lignes_global = ["A", "B", "C"]
noms_colonnes_global = ["D", "E", "F"]
trace, viv_l, viv_c = idsds_cardinal_trace(G0_P1, G0_P2, [0, 1, 2], [0, 1, 2])
for ligne in trace:
    print(ligne)
print()
print("Strategies rationalisables :",
      [noms_lignes_global[i] for i in viv_l], "x", [noms_colonnes_global[j] for j in viv_c])

etape 0 : Joueur 1 supprime C (dominee par la mixte {'A': 0.5, 'B': 0.5, 'C': 0.0}, marge 0.5000)
etape 1 : Joueur 2 supprime F (dominee par la mixte {'D': 0.5, 'E': 0.5, 'F': 0.0}, marge 0.5000)
etape 2 : Joueur 1 supprime B (dominee par la mixte {'A': 1.0, 'B': 0.0}, marge 1.0000)
etape 3 : Joueur 2 supprime E (dominee par la mixte {'D': 1.0, 'E': 0.0}, marge 3.0000)

Strategies rationalisables : ['A'] x ['D']


In [5]:
# Controle : le profil rationalisable unique est-il l'equilibre de Nash unique de G0 ?
jeu_G0 = nash.Game(G0_P1, G0_P2)
equilibres = list(jeu_G0.support_enumeration())
print("equilibres de Nash de G0 (support enumeration) :")
for eq in equilibres:
    i = int(np.argmax(eq[0])); j = int(np.argmax(eq[1]))
    print(f"  ({noms_lignes_global[i]}, {noms_colonnes_global[j]})  mixtes P1={np.round(eq[0], 6)} P2={np.round(eq[1], 6)}")
print()
print("verdict : ensemble rationalisable == ensemble des equilibria support :",
      sorted((int(np.argmax(e[0])), int(np.argmax(e[1]))) for e in equilibres) == sorted((i, j) for i in viv_l for j in viv_c))

equilibres de Nash de G0 (support enumeration) :
  (A, D)  mixtes P1=[1. 0. 0.] P2=[1. 0. 0.]

verdict : ensemble rationalisable == ensemble des equilibria support : True


### Interprétation : rationalisables uniques ⇒ équilibre unique

Les quatre suppressions recalculées épousent la trace du manuel — $C$ chassée par $\frac{1}{2}A + \frac{1}{2}B$ (marge $0{,}5$ sur chaque colonne), puis $F$ par $\frac{1}{2}D + \frac{1}{2}E$, puis $B$ par $A$ (dominance pure), puis $E$ par $D$ — et le survivant est le singleton $(A, D)$, exactement le seul équilibre que `nashpy` énumère. Dans ce jeu, la rationalité itérée **suffit** à prédire l'issue : l'hypothèse d'équilibre n'ajoute rien. La section suivante mesure à quel point ce cas est exceptionnel.

## 4. Le pont vers le palier 04 : ce que la rationalité ne délie pas

Les quatre jeux emblématiques du palier 04 — Dilemme du Prisonnier, Battle of the Sexes, chasse au cerf, Matching Pennies — passent au même banc : ensembles rationalisables (IDSDS cardinal) contre équilibres de Nash (mixtes compris). La question : l'inclusion est-elle stricte ?

In [6]:
# Les jeux du palier 04 (memes matrices que 04i section 6).
jeux_palier04 = {
    "Dilemme du Prisonnier": (
        np.array([[3.0, 0.0], [5.0, 1.0]]), np.array([[3.0, 5.0], [0.0, 1.0]]),
        ["Cooperer", "Defaire"], ["Cooperer", "Defaire"]),
    "Battle of the Sexes": (
        np.array([[3.0, 0.0], [0.0, 2.0]]), np.array([[2.0, 0.0], [0.0, 3.0]]),
        ["Opera", "Football"], ["Opera", "Football"]),
    "Chasse au cerf": (
        np.array([[4.0, 0.0], [3.0, 3.0]]), np.array([[4.0, 3.0], [0.0, 3.0]]),
        ["Cerf", "Lievre"], ["Cerf", "Lievre"]),
    "Matching Pennies": (
        np.array([[1.0, -1.0], [-1.0, 1.0]]), np.array([[-1.0, 1.0], [1.0, -1.0]]),
        ["Pile", "Face"], ["Pile", "Face"]),
}

def rationalisables_jeu_2x2(P1, P2):
    """Index des lignes / colonnes survivant a l'IDSDS cardinal sur un jeu 2x2 (memo)."""
    viv_l, viv_c = [0, 1], [0, 1]
    while True:
        supprime = False
        M = P1[np.ix_(viv_l, viv_c)]
        for k in range(M.shape[0]):
            t, _ = marge_dominee_par_mixte(M, k)
            if t > 1e-9:
                viv_l.pop(k); supprime = True; break
        if not supprime:
            M2 = P2[np.ix_(viv_l, viv_c)].T
            for k in range(M2.shape[0]):
                t, _ = marge_dominee_par_mixte(M2, k)
                if t > 1e-9:
                    viv_c.pop(k); supprime = True; break
        if not supprime:
            break
    return viv_l, viv_c

print(f"{'jeu':>24} | {'rationalisables':>34} | {'equilibres (nashpy)':>28} | verdict")
print("-" * 100)
for nom, (P1, P2, nl, nc) in jeux_palier04.items():
    viv_l, viv_c = rationalisables_jeu_2x2(P1, P2)
    rat = [nl[i] for i in viv_l], [nc[j] for j in viv_c]
    eqs = list(nash.Game(P1, P2).support_enumeration())
    purs = [(nl[int(np.argmax(e[0]))], nc[int(np.argmax(e[1]))]) for e in eqs
            if np.max(e[0]) > 1 - 1e-9 and np.max(e[1]) > 1 - 1e-9]
    n_mixtes = len(eqs) - len(purs)
    col_eq = f"{len(purs)} pur(s) {purs} + {n_mixtes} mixte(s)"
    tout_survit = (len(viv_l) == 2 and len(viv_c) == 2)
    verdict = "inclusion stricte" if tout_survit else "purge -> singleton = equilibre"
    print(f"{nom:>24} | {str(rat[0]) + ' x ' + str(rat[1]):>34} | {col_eq:>28} | {verdict}")

                     jeu |                    rationalisables |          equilibres (nashpy) | verdict
----------------------------------------------------------------------------------------------------
   Dilemme du Prisonnier |          ['Defaire'] x ['Defaire'] | 1 pur(s) [('Defaire', 'Defaire')] + 0 mixte(s) | purge -> singleton = equilibre
     Battle of the Sexes | ['Opera', 'Football'] x ['Opera', 'Football'] | 2 pur(s) [('Opera', 'Opera'), ('Football', 'Football')] + 1 mixte(s) | inclusion stricte
          Chasse au cerf | ['Cerf', 'Lievre'] x ['Cerf', 'Lievre'] | 2 pur(s) [('Cerf', 'Cerf'), ('Lievre', 'Lievre')] + 1 mixte(s) | inclusion stricte
        Matching Pennies | ['Pile', 'Face'] x ['Pile', 'Face'] |     0 pur(s) [] + 1 mixte(s) | inclusion stricte


### Interprétation : l'inclusion stricte est la norme, pas l'exception

Le tableau le dit en une colonne : seul le **Dilemme du Prisonnier** voit la rationalité itérée refermer l'équilibre (la dominance de Défaire purge les deux lignes/colonnes, et le survivant unique *est* l'équilibre) ; sur les trois autres jeux, **aucune** stratégie n'est dominée — même par une mixte — donc l'ensemble rationalisable reste le produit complet des stratégies, tandis que les équilibres sont trois points (BoS), deux points (chasse au cerf) ou un point mixte (Matching Pennies). L'écart mesuré entre les deux colonnes est exactement ce que l'hypothèse d'équilibre **ajoute** à la rationalité commune : la cohérence mutuelle des conjectures, chacune étant *confirmée* par la stratégie de l'autre — la lecture « mixte = croyance » du carnet 04f.

## 5. La lecture épistémique : rationalité commune de croyance

La **remarque 5.5** du manuel donne la définition opératoire : un joueur est *rationnel* si sa stratégie pure est une meilleure réponse à sa croyance sur l'adversaire ; à deux joueurs, une croyance est une distribution — le même objet qu'une stratégie mixte. En itérant — « je crois que tu es rationnel, tu crois que je crois que tu es rationnel, … » — on obtient la **rationalité commune de croyance**, et le manuel annonce (ch. 9, développé au carnet 02d) le théorème : sous rationalité commune, seuls des profils rationalisables peuvent être joués.

La **remarque 5.6** pose la limite, et elle n'est pas décorative : l'itération exige que les préférences **cardinales** (vNM) des joueurs soient connaissance commune. Un adversaire qui ne connaîtrait que le *classement ordinal* des issues ne pourrait pas déduire que la stratégie $B$ de la Table 5.8 est irrationnelle — la mixte $
rac{1}{3}A + 
rac{2}{3}C$ qui la condamne n'existe que dans les valeurs, pas dans l'ordre. Exiger la connaissance commune d'utilités cardinales est une hypothèse forte (le manuel lui-même la dit « souvent très irréaliste ») : la rationalisabilité hérite exactement de cette fragilité, et c'est un enseignement du carnet autant qu'un théorème.

## 6. Résumé

| Concept | Énoncé | Où il est mesuré |
|---|---|---|
| Dominance par une mixte | $\frac{1}{3}A + \frac{2}{3}C$ rapporte $\frac{4}{3}$ contre les deux colonnes, B en rapporte 1 : B condamnée sans dominance pure | Section 1, Table 5.8 |
| Théorème de Pearce (1984) | jamais-meilleure-réponse $\iff$ dominée par une mixte (2 joueurs) | Section 2 : deux LP indépendants, 3/3 accords, mixte $(\frac{1}{3}, 0, \frac{2}{3})$ retrouvée par le solveur |
| IDSDS cardinal | itérer la suppression des dominées-par-mixte ; survivants = rationalisables | Section 3 : Figure 5.9 rejouée, 4 suppressions, survivant $(A, D)$ |
| Rationalisables $\supseteq$ équilibres | stricte dès qu'aucune dominance n'amorce la purge | Section 4 : PD égalité ; BoS, cerf, pennies inclusion stricte |
| Condition épistémique | rationalité commune de croyance $\Rightarrow$ profils rationalisables ; exige les vNM en connaissance commune | Section 5 (renvois 02d, 04f) |

## 7. Exercices

Trois exercices pour s'approprier la machinerie. Les stubs s'exécutent sans erreur (règle C.1) : remplacez `pass` par votre code, la cellule de contrôle restante doit rester verte. Les exercices 1 et 2 reprennent les exercices 5.12 et 5.13 du manuel (Bonanno, Appendix 5.E.4) ; l'exercice 3 en est le 5.14, à trois joueurs.

### Exercice 1 : IDSDS cardinal sur une 3×3 (Bonanno, exercice 5.12)

Le jeu ci-dessous est celui de l'exercice 5.12 du manuel. Appliquez l'IDSDS cardinal et retournez les ensembles rationalisables de chaque joueur sous forme de listes de noms de stratégies.

In [7]:
P1_512 = np.array([[3.0, 2.0, 2.0],
                   [5.0, 1.0, 2.0],
                   [9.0, 1.0, 3.0]])
P2_512 = np.array([[5.0, 0.0, 2.0],
                   [2.0, 2.0, 1.0],
                   [0.0, 5.0, 2.0]])
noms_512_P1 = ["A", "B", "C"]; noms_512_P2 = ["L", "M", "R"]

def exercice_1_idscds_512() -> dict:
    """Retourner {"P1": [...noms rationalisables...], "P2": [...]} pour le jeu 5.12."""
    pass  # TODO etudiant : iterer marge_dominee_par_mixte sur les deux joueurs et collecter les survivants
    # Indice : la fonction idsds_cardinal_trace de la section 3 fait exactement ce travail.

print("Exercice a completer : exercice_1_idscds_512 doit retourner les ensembles rationalisables du jeu 5.12.")

Exercice a completer : exercice_1_idscds_512 doit retourner les ensembles rationalisables du jeu 5.12.


### Exercice 2 : vrai ou faux — mixtures de rationalisables (Bonanno, exercice 5.13)

L'énoncé à tester : *« Si A et B sont deux stratégies rationalisables du Joueur 1, alors toute stratégie mixte qui met une probabilité positive sur A et B (et nulle ailleurs) est une meilleure réponse à quelque stratégie mixte du Joueur 2. »* Construisez un contre-exemple : un jeu 2×2 (ou plus grand) où A et B survivent toutes les deux à l'IDSDS cardinal, mais où une mixture donnée de A et B n'est meilleure réponse à **aucune** croyance de l'adversaire.

In [8]:
def exercice_2_contre_exemple_513() -> dict:
    """Retourner {"P1": matrice numpy, "P2": matrice numpy, "mixture": numpy, "explication": str}.
    La mixture porte sur deux lignes toutes deux rationalisables, et n'est jamais meilleure reponse
    (verifiable : marge_meilleure_reponse sur chaque ligne du jeu mixte... a vous de l'adapter)."""
    pass  # TODO etudiant : construire le jeu, la mixture, et une phrase d'explication
    # Indice : cherchez un jeu ou une ligne est faiblement dominee mais pas strictement.

print("Exercice a completer : exercice_2_contre_exemple_513 doit retourner le jeu, la mixture et l'explication.")

Exercice a completer : exercice_2_contre_exemple_513 doit retourner le jeu, la mixture et l'explication.


### Exercice 3 : à trois joueurs, la croyance corrélée (Bonanno, exercice 5.14)

Le jeu à trois joueurs de l'exercice 5.14 du manuel, où seuls les paiements du Joueur 1 sont montrés — le Joueur 1 choisit la ligne (A, B, C, D), le Joueur 2 la colonne (E, F) et le Joueur 3 le panneau (G, H) :

| P1 \ P2 | E | F | | E | F |
|---|---|---|---|---|---|
| **A** | 3 | 0 | **A** | 0 | 0 |
| **B** | 0 | 3 | **B** | 3 | 0 |
| **C** | 0 | 0 | **C** | 0 | 3 |
| **D** | 2 | 0 | **D** | 0 | 2 |

(le panneau de gauche correspond à « Joueur 3 joue G », celui de droite à « Joueur 3 joue H »)

1. Vérifiez que si le Joueur 1 croit à ½ que l'événement « J2 joue E **et** J3 joue G » se réalisera, et à ½ que « J2 joue F **et** J3 joue H » se réalisera, alors **D** est sa meilleure réponse.
2. Montrez qu'il n'existe **aucun** profil mixte **indépendant** $(p, 1{-}p)$ sur (E, F) et $(q, 1{-}q)$ sur (G, H) des Joueurs 2 et 3 contre lequel D soit une meilleure réponse de P1 — c'est la démonstration que le théorème de Pearce ne s'étend pas tel quel au-delà de deux joueurs : une croyance peut être **corrélée**, un profil mixte ne peut pas.

In [9]:
# Panneaux du jeu 5.14 (paiements du Joueur 1) : lignes A, B, C, D ; colonnes E, F.
panneau_G_514 = np.array([[3.0, 0.0],
                          [0.0, 3.0],
                          [0.0, 0.0],
                          [2.0, 0.0]])   # Joueur 3 joue G
panneau_H_514 = np.array([[0.0, 0.0],
                          [3.0, 0.0],
                          [0.0, 3.0],
                          [0.0, 2.0]])   # Joueur 3 joue H

def exercice_3_croyance_correlee_514() -> dict:
    """Retourner {"croyance_correlee": np.ndarray 2x2, "meilleure_marge_independante": float, "explication": str}.
    croyance_correlee[i, j] = probabilite de (J2 joue E si i=0, F si i=1 ; J3 joue G si j=0, H si j=1),
    qui doit faire de D une meilleure reponse.
    meilleure_marge_independante = max sur (p, q) de [u(D) - max(u(A), u(B), u(C))] contre le produit independant,
    a verifier strictement negative."""
    pass  # TODO etudiant : construire la croyance correlee 2x2 et balayer la grille (p, q)
    # Indice : contre un produit independant (p, q), poser s = p + q - 2pq (probabilite des evenements
    # "hors diagonale") et montrer que u(D) >= u(B) exige s <= 2/5 alors que u(D) >= u(A) et u(D) >= u(C)
    # exigent pq et (1-p)(1-q) dans un rapport <= 2, donc s >= 6*sqrt(2) - 8 (environ 0,485) : contradiction.

print("Exercice a completer : exercice_3_croyance_correlee_514 doit retourner la croyance correlee et la marge.")

Exercice a completer : exercice_3_croyance_correlee_514 doit retourner la croyance correlee et la marge.


## Références académiques

1. **Pearce, D. (1984).** *Rationalizable strategic behavior and the problem of perfection.* Econometrica — origine du théorème 5.3, cité et énoncé au ch. 5.4 du manuel ; l'énoncé et la preuve suivis ici sont ceux de la copie archivée de Bonanno (2015).
2. **Bonanno, G. (2015).** *Game Theory* (open access textbook, 165 solved exercises), ch. 5.4 « Strict dominance and rationalizability » (p. 201-204), Table 5.8, théorème 5.3, définition 5.6, Figure 5.9, remarques 5.5-5.6, exercices 5.12-5.14. arXiv:1512.06808. Copie archivée au gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\`.
3. **Von Neumann, J. & Morgenstern, O. (1944).** *Theory of Games and Economic Behavior.* Princeton University Press — les paiements cardinaux que toute la section 1 suppose, construits au carnet 04i.

***